In [ ]:
import os
import time
from PIL import Image
from PIL.ExifTags import TAGS, GPSTAGS
from collections import defaultdict
from geopy.geocoders import Nominatim

# 住所取得のための初期設定 (user_agentは任意の名前でOKです)
geolocator = Nominatim(user_agent="exif_photo_sorter")

def get_address(lat, lon):
    """緯度経度から大まかな住所を取得する"""
    try:
        # OpenStreetMapのサーバーに負荷をかけないよう、1秒待機する（必須マナー）
        time.sleep(1)
        # language='ja' で日本語の住所を取得
        location = geolocator.reverse(f"{lat}, {lon}", language='ja')
        if location:
            # 取得した住所の文字列を返す
            return location.address
        return "住所不明"
    except Exception as e:
        return f"住所取得エラー"

def get_exif_data(image_path):
    try:
        with Image.open(image_path) as img:
            exif = img._getexif()
            if not exif:
                return None
            return {TAGS.get(key, key): val for key, val in exif.items()}
    except Exception as e:
        print(f"エラー ({image_path}): {e}")
        return None

def get_gps_info(exif_data):
    if 'GPSInfo' not in exif_data:
        return None
    gps_info = {}
    for key, val in exif_data['GPSInfo'].items():
        decode_name = GPSTAGS.get(key, key)
        gps_info[decode_name] = val
    return gps_info

def convert_to_decimal(dms_value, ref):
    try:
        degrees = float(dms_value[0])
        minutes = float(dms_value[1])
        seconds = float(dms_value[2])
        decimal = degrees + (minutes / 60.0) + (seconds / 3600.0)
        if ref in ['S', 'W']:
            decimal = -decimal
        return decimal
    except Exception:
        return None

def get_coordinates(gps_info):
    if not gps_info:
        return None, None
    lat, lon = None, None
    if 'GPSLatitude' in gps_info and 'GPSLatitudeRef' in gps_info:
        lat = convert_to_decimal(gps_info['GPSLatitude'], gps_info['GPSLatitudeRef'])
    if 'GPSLongitude' in gps_info and 'GPSLongitudeRef' in gps_info:
        lon = convert_to_decimal(gps_info['GPSLongitude'], gps_info['GPSLongitudeRef'])
    return lat, lon

def classify_photos_by_location(target_folder):
    location_groups = defaultdict(list)
    no_gps_files = []

    print(f"[{target_folder}] の解析を開始します...\n")

    for filename in os.listdir(target_folder):
        if not filename.lower().endswith(('.jpg', '.jpeg')):
            continue

        filepath = os.path.join(target_folder, filename)
        exif_data = get_exif_data(filepath)

        if exif_data:
            gps_info = get_gps_info(exif_data)
            lat, lon = get_coordinates(gps_info)

            if lat is not None and lon is not None:
                grid_size = 0.003  # 約300m
                group_lat = round((lat // grid_size) * grid_size, 3)
                group_lon = round((lon // grid_size) * grid_size, 3)
                group_key = (group_lat, group_lon)
                location_groups[group_key].append({
                    'filename': filename,
                    'lat': lat,
                    'lon': lon
                })
            else:
                no_gps_files.append(filename)
        else:
            no_gps_files.append(filename)

    print("=== 分類結果 ===")
    for (group_lat, group_lon), files in location_groups.items():
        print(f"\n📍 エリア: 緯度 {group_lat:.3f}, 経度 {group_lon:.3f} (付近)")
        
        # 代表として、グループの座標から大まかな住所を取得して表示
        address = get_address(group_lat, group_lon)
        print(f"   推定住所: {address}")
        
        print(f"   該当ファイル数: {len(files)}枚")
        for f_info in files:
            print(f"    - {f_info['filename']} (詳細座標: {f_info['lat']:.5f}, {f_info['lon']:.5f})")

    if no_gps_files:
        print(f"\n⚠️ 位置情報がない、または取得できなかったファイル ({len(no_gps_files)}枚):")
        for filename in no_gps_files:
            print(f"    - {filename}")

# ==========================================
# 実行部分
# ==========================================
FOLDER_PATH = r"\\位置分析したい写真ファイルがあるフォルダのパス"

if os.path.exists(FOLDER_PATH):
    classify_photos_by_location(FOLDER_PATH)
else:
    print(f"エラー: フォルダ '{FOLDER_PATH}' が見つかりません。")

[\\202.251.23.197\業務\26\PHOTO(ORG）現場写真記録\M17026023_滋賀国電通26\260701～0703_NW現地踏査（大越編集前）\※山本撮影(彦根→柏原→木ノ本→沓掛→国境)] の解析を開始します...

=== 分類結果 ===

📍 エリア: 緯度 35.265, 経度 136.266 (付近)
   推定住所: 国道8号, Ekihigashi-cho, 彦根, 彦根市, 滋賀県, 522-8501, 日本
   該当ファイル数: 378枚
    - 20260702_084637.jpg (詳細座標: 35.26539, 136.26820)
    - 20260702_084922.jpg (詳細座標: 35.26529, 136.26815)
    - 20260702_084931.jpg (詳細座標: 35.26525, 136.26813)
    - 20260702_085125.jpg (詳細座標: 35.26521, 136.26830)
    - 20260702_085127.jpg (詳細座標: 35.26521, 136.26830)
    - 20260702_090006.jpg (詳細座標: 35.26523, 136.26832)
    - 20260702_090042.jpg (詳細座標: 35.26524, 136.26832)
    - 20260702_090046.jpg (詳細座標: 35.26524, 136.26832)
    - 20260702_090055.jpg (詳細座標: 35.26524, 136.26832)
    - 20260702_090059.jpg (詳細座標: 35.26524, 136.26832)
    - 20260702_090101.jpg (詳細座標: 35.26524, 136.26832)
    - 20260702_090104.jpg (詳細座標: 35.26524, 136.26832)
    - 20260702_090106.jpg (詳細座標: 35.26524, 136.26832)
    - 20260702_090108.jpg (詳細座標: 35.26524, 136.26832